# AI Models Server for Social Media App

Welcome! This notebook will run the AI Flask server for your **Social Media App** on Google Colab and expose it to the public internet using **Ngrok (Credit-Card-Free & Persistent)** or **Cloudflare Tunnel**.

### How to run:
1. **Enable GPU (Recommended)**: Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` -> Click `Save`.
2. **Set up a Persistent Tunnel (Recommended)**: In **Step 4**, enter your free **Ngrok Authtoken & Domain** (No credit card needed!) or **Cloudflare Token**.
3. Run all the cells in this notebook (`Runtime` -> `Run all` or `Ctrl + F9`).
4. Copy your generated or persistent URL shown in Step 4.
5. Update the `baseUrl` inside `lib/core/services/api_service.dart` in your Flutter project with this URL.
6. Run Step 5 to view requests, object detections, captions, and deepfake results in real-time!

### Step 1: Install Dependencies
We will install Flask, Flask-CORS, Ultralytics (YOLOv8), and Hugging Face Transformers.

In [ ]:
!pip install Flask flask-cors ultralytics transformers torch pillow

### Step 2: Write Python Server Files
We write the `app.py` script directly to the Colab environment, complete with detailed print logging.

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image, ImageDraw
import io, os, json, base64
import torch
from ultralytics import YOLO
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer, pipeline

app = Flask(__name__)
CORS(app)

import logging
# Flask Werkzeug requests logs (Enabled)
log = logging.getLogger('werkzeug')
log.setLevel(logging.INFO)

@app.before_request
def log_request_info():
    print(f"\\n=========================================")
    print(f"[HTTP REQUEST] ---> Received {request.method} request to {request.path}")
    if request.files:
        print(f"Files attached: {list(request.files.keys())}")

@app.after_request
def log_response_info(response):
    print(f"[HTTP RESPONSE] <--- Sent {response.status_code} response for {request.path}")
    print(f"=========================================\\n")
    return response

# Suppress Transformers warning logs
logging.getLogger("transformers").setLevel(logging.ERROR)

# API Key
API_KEY = "social_media_app_2024_secure_key"

print("Loading AI Models...")

# Model 1: Object Detection
obj_model = YOLO('yolov8n.pt')

# Model 2: Image Captioning
cap_model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
cap_processor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
cap_tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")

# Model 3: Deepfake Detection
fake_pipe = pipeline("image-classification", model="umm-maybe/AI-image-detector")

print("All Models Ready!\n")

UPLOAD_FOLDER = 'uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

def check_api_key():
    key = request.headers.get('X-API-Key')
    if key != API_KEY:
        return False
    return True

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'message': 'AI Server Running'})

@app.route('/detect_objects', methods=['POST'])
def detect_objects():
    print("\n--- NEW OBJECT DETECTION REQUEST ---")
    if not check_api_key():
        print("Error: Invalid API Key")
        return jsonify({'error': 'Invalid API Key'}), 401
    
    if 'image' not in request.files:
        print("Error: No image provided")
        return jsonify({'error': 'No image provided'}), 400
    
    file = request.files['image']
    print(f"Processing image: {file.filename}")
    img = Image.open(file).convert('RGB')
    draw = ImageDraw.Draw(img)
    
    print("--- Analyze Image ---")
    print("--- Detect Objects ---")
    print("Running YOLOv8 model...")
    results = obj_model(img, verbose=False)
    result = results[0]
    
    detections = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        name = obj_model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1-15), f"{name} ({conf*100:.1f}%)", fill="red")
        
        detections.append({
            "object": name,
            "label": name,
            "confidence": round(conf*100, 1),
            "bbox": {"x1": int(x1), "y1": int(y1), "x2": int(x2), "y2": int(y2)}
        })
    
    print("\n--- RESULTS ---")
    print(f"Objects Detected: {[d['object'] for d in detections]}")
    print("--------------------\n")
    
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    encoded = base64.b64encode(buffered.getvalue()).decode()
    
    return jsonify({
        'total_objects': len(detections),
        'detections': detections,
        'objects': [d['object'] for d in detections],
        'marked_image': encoded
    })

@app.route('/generate_caption', methods=['POST'])
def generate_caption():
    print("\n--- NEW IMAGE CAPTION REQUEST ---")
    if not check_api_key():
        print("Error: Invalid API Key")
        return jsonify({'error': 'Invalid API Key'}), 401
    
    if 'image' not in request.files:
        print("Error: No image provided")
        return jsonify({'error': 'No image provided'}), 400
    
    file = request.files['image']
    print(f"Processing image: {file.filename}")
    img = Image.open(file).convert('RGB')
    
    print("--- Captions ---")
    print("Running ViT-GPT2 Image Captioning...")
    pixel_values = cap_processor(images=img, return_tensors="pt").pixel_values
    
    with torch.no_grad():
        output_ids = cap_model.generate(pixel_values, max_length=16, num_beams=4)
    
    caption = cap_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("\n--- RESULTS ---")
    print(f"Generated Caption: '{caption}'")
    print("--------------------\n")
    
    return jsonify({'caption': caption})

@app.route('/detect_deepfake', methods=['POST'])
def detect_deepfake():
    print("\n--- NEW DEEPFAKE DETECTION REQUEST ---")
    if not check_api_key():
        print("Error: Invalid API Key")
        return jsonify({'error': 'Invalid API Key'}), 401
    
    if 'image' not in request.files:
        print("Error: No image provided")
        return jsonify({'error': 'No image provided'}), 400
    
    file = request.files['image']
    print(f"Processing image: {file.filename}")
    img = Image.open(file).convert('RGB')
    
    print("Running Deepfake classifier...")
    results = fake_pipe(img)
    highest_class = max(results, key=lambda x: x['score'])
    is_real = 'human' in highest_class['label'].lower() or 'real' in highest_class['label'].lower()
    confidence = float(highest_class['score'])
    
    # Calculate Real vs AI percentages
    real_percent = 0.0
    ai_percent = 0.0
    for r in results:
        label_lower = r['label'].lower()
        if 'human' in label_lower or 'real' in label_lower:
            real_percent = float(r['score']) * 100
        elif 'artificial' in label_lower or 'ai' in label_lower or 'fake' in label_lower:
            ai_percent = float(r['score']) * 100
            
    # Fallback/validation to ensure they sum to ~100
    if real_percent == 0.0 and ai_percent == 0.0:
        if is_real:
            real_percent = confidence * 100
            ai_percent = 100.0 - real_percent
        else:
            ai_percent = confidence * 100
            real_percent = 100.0 - ai_percent
    elif real_percent == 0.0:
        real_percent = max(0.0, 100.0 - ai_percent)
    elif ai_percent == 0.0:
        ai_percent = max(0.0, 100.0 - real_percent)
        
    print("\n--- RESULTS ---")
    print(f"Deepfake Verdict: {'Real' if is_real else 'AI-Generated'} (Real: {real_percent:.1f}%, AI: {ai_percent:.1f}%)")
    print("--------------------\n")
    
    return jsonify({
        'is_real': is_real,
        'confidence': confidence,
        'real_percent': round(real_percent, 1),
        'ai_percent': round(ai_percent, 1),
        'results': [{'label': r['label'], 'score': round(r['score']*100, 1)} for r in results]
    })

@app.route('/process_all', methods=['POST'])
def process_all():
    """Ek hi call mein teeno models ka result"""
    print("\n--- NEW BATCH PROCESS ALL REQUEST ---")
    if not check_api_key():
        print("Error: Invalid API Key")
        return jsonify({'error': 'Invalid API Key'}), 401
    
    if 'image' not in request.files:
        print("Error: No image provided")
        return jsonify({'error': 'No image provided'}), 400
    
    file = request.files['image']
    print(f"Processing image: {file.filename}")
    img = Image.open(file).convert('RGB')
    
    # Object Detection
    print("--- Analyze Image ---")
    print("--- Detect Objects ---")
    print("1. Running YOLOv8 Object Detection...")
    draw = ImageDraw.Draw(img)
    obj_results = obj_model(img, verbose=False)
    detections = []
    for box in obj_results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        name = obj_model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1-15), f"{name} ({conf*100:.1f}%)", fill="red")
        detections.append({
            "object": name,
            "label": name,
            "confidence": round(conf*100, 1)
        })
    print("\n--- BATCH RESULTS ---")
    print(f"Objects Detected: {[d['object'] for d in detections]}")
    
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    marked_image = base64.b64encode(buffered.getvalue()).decode()
    
    # Caption
    print("--- Captions ---")
    print("2. Running ViT-GPT2 Image Captioning...")
    pixel_values = cap_processor(images=img, return_tensors="pt").pixel_values
    with torch.no_grad():
        output_ids = cap_model.generate(pixel_values, max_length=16, num_beams=4)
    caption = cap_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Generated Caption: '{caption}'")
    
    # Deepfake
    print("3. Running Deepfake Detection...")
    fake_results = fake_pipe(img)
    highest_class = max(fake_results, key=lambda x: x['score'])
    is_real = 'human' in highest_class['label'].lower() or 'real' in highest_class['label'].lower()
    confidence = float(highest_class['score'])
    
    # Calculate Real vs AI percentages
    real_percent = 0.0
    ai_percent = 0.0
    for r in fake_results:
        label_lower = r['label'].lower()
        if 'human' in label_lower or 'real' in label_lower:
            real_percent = float(r['score']) * 100
        elif 'artificial' in label_lower or 'ai' in label_lower or 'fake' in label_lower:
            ai_percent = float(r['score']) * 100
            
    if real_percent == 0.0 and ai_percent == 0.0:
        if is_real:
            real_percent = confidence * 100
            ai_percent = 100.0 - real_percent
        else:
            ai_percent = confidence * 100
            real_percent = 100.0 - ai_percent
    elif real_percent == 0.0:
        real_percent = max(0.0, 100.0 - ai_percent)
    elif ai_percent == 0.0:
        ai_percent = max(0.0, 100.0 - real_percent)
        
    print(f"Deepfake Verdict: {'Real' if is_real else 'AI-Generated'} (Real: {real_percent:.1f}%, AI: {ai_percent:.1f}%)")
    print("--------------------\n")
    
    return jsonify({
        'objects': detections,
        'marked_image': marked_image,
        'caption': caption,
        'deepfake': {
            'is_real': is_real,
            'confidence': confidence,
            'real_percent': round(real_percent, 1),
            'ai_percent': round(ai_percent, 1),
            'results': [{'label': r['label'], 'score': round(r['score']*100, 1)} for r in fake_results]
        }
    })

if __name__ == '__main__':
    print("\n🚀 Server: http://0.0.0.0:5000")
    print(f"🔑 API Key: {API_KEY}")
    app.run(host='0.0.0.0', port=5000, debug=False)

### Step 3: Run Flask Server in Background
We run `app.py` asynchronously and stream outputs to a log file.

In [ ]:
import subprocess
import time

print("Starting Flask server in the background...")
with open("flask_server.log", "w") as log_file:
    server_process = subprocess.Popen(["python", "app.py"], stdout=log_file, stderr=log_file)

print("Waiting 15 seconds for models to load in memory...")
time.sleep(15)
print("Flask server process initiated. Checking logs:")

with open("flask_server.log", "r") as log_file:
    print(log_file.read())

### Step 4: Launch Persistent or Quick Tunnel
Configure one of the persistent tunnel options below to get a static URL that **never changes** on server restarts:
- **Option A (Ngrok)**: Recommended, 100% free, and does **NOT** require any credit card details.
- **Option B (Cloudflare)**: Requires a free Cloudflare account with a valid payment method set up.
- **Fallback (TryCloudflare)**: If both are left blank, a random quick tunnel URL will be generated.

In [ ]:
# Download cloudflared CLI and install ngrok
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!pip install pyngrok --quiet

# ==============================================================================
# 🔑 CHOOSE YOUR PERSISTENT TUNNEL OPTION (No Credit Card Required!)
# ==============================================================================
# OPTION A: NGROK (Recommended - 100% Free & No Credit Card Needed)
# 1. Create a free account at https://ngrok.com (absolutely no credit card requested).
# 2. Copy your "Authtoken" from the Ngrok dashboard.
# 3. Copy your 1 free "Static Domain" (e.g., your-domain.ngrok-free.app) from the dashboard.
NGROK_AUTHTOKEN = "3E6uCYTOIwLKpu6Og19WcYgeJHR_2EzCmDF29YpsZ4A4PbDMg"
NGROK_STATIC_DOMAIN = "eatable-monument-gone.ngrok-free.dev"

# OPTION B: CLOUDFLARE ZERO TRUST (Requires Cloudflare Account & Payment Card)
CLOUDFLARE_TUNNEL_TOKEN = ""
# ==============================================================================

# Start tunnel
import subprocess
import re
import time

print("Launching Tunnel...")
with open("tunnel.log", "w") as log_file:
    if NGROK_AUTHTOKEN.strip() and NGROK_STATIC_DOMAIN.strip():
        print("Using Persistent Ngrok Tunnel...")
        # Configure authtoken
        subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN.strip()], stdout=log_file, stderr=log_file)
        # Run tunnel
        tunnel_process = subprocess.Popen(["ngrok", "http", "5000", "--domain", NGROK_STATIC_DOMAIN.strip()], stdout=log_file, stderr=log_file)
        
        time.sleep(5)
        print("\n" + "="*60)
        print("🎉 PERSISTENT NGROK TUNNEL RUNNING!")
        print(f"🔗 PUBLIC URL: https://{NGROK_STATIC_DOMAIN.strip()}")
        print("="*60 + "\n")
        print("Copy the URL above and paste it into lib/core/services/api_service.dart baseUrl.")
        
    elif CLOUDFLARE_TUNNEL_TOKEN.strip():
        print("Using Persistent Cloudflare Tunnel...")
        tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN.strip()], stdout=log_file, stderr=log_file)
        
        time.sleep(5)
        print("\n" + "="*60)
        print("🎉 PERSISTENT CLOUDFLARE TUNNEL RUNNING!")
        print("🔗 The API requests will route to your custom Cloudflare Domain.")
        print("="*60 + "\n")
        
    else:
        print("Using Free TryCloudflare Quick Tunnel (Link will change on restart)...\n")
        tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:5000"], stdout=log_file, stderr=log_file)
        
        time.sleep(5)
        # Search for trycloudflare.com URL in the logs
        with open("tunnel.log", "r") as log_file:
            log_content = log_file.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
            if match:
                public_url = match.group(0)
                print("\n" + "="*60)
                print("🎉 SERVER SUCCESSFULLY RUNNING!")
                print(f"🔗 PUBLIC URL: {public_url}")
                print("="*60 + "\n")
                print("Copy the URL above and paste it into lib/core/services/api_service.dart baseUrl.")
            else:
                print("Tunnel failed to start or URL not found. Tunnel Logs:")
                print(log_content)


### Step 5: Real-time Log Viewer
Run this cell to view requests and processing logs in real-time as you interact with the Flutter app! (Press the stop button to exit log viewing).

In [ ]:
import time
import os

print("Streaming Flask Server Logs in real-time (Press the Cell Stop button to stop viewing logs):\n")
log_path = "flask_server.log"
if os.path.exists(log_path):
    with open(log_path, "r") as f:
        # Seek to end
        # f.seek(0, 2) # Commented out so it shows old logs too\n
        try:
            while True:
                line = f.readline()
                if not line:
                    time.sleep(0.3)
                    continue
                print(line, end="")
        except KeyboardInterrupt:
            print("\nStopped streaming logs.")
else:
    print(f"Log file '{log_path}' not found!")